# keys

> Keyboard access outside the browser (macOS only).

In [ ]:
#| default_exp keys

In [ ]:
#| hide
from nbdev.showdoc import *

A web page can only see keys while its own tab is focused and can't type into other apps. This server runs on the user's Mac, so it does both, with two macOS permissions that are granted to the app that launched it (Terminal, iTerm, Cursor...), not to the web app itself:

- **Accessibility** lets us post keystrokes into the focused app (`type_key`)
- **Input Monitoring** lets us see the hold-to-talk key in other apps (`watch`)

In [ ]:
#| export
import asyncio, subprocess, sys, threading

In [ ]:
#| export
try:
    import Quartz as Q
    from ApplicationServices import AXIsProcessTrusted, AXIsProcessTrustedWithOptions
    available = sys.platform == 'darwin'
except Exception: available = False

In [ ]:
#| export
CODES = dict(space=49, **{'return':36}, tab=48, backspace=51, escape=53, left=123, right=124, down=125, up=126)
# modifier keys the page can use as the hold key: key code -> (the page's name for it, flag that is set while it is down)
MODS = {61: ('AltRight', 'kCGEventFlagMaskAlternate'), 54: ('MetaRight', 'kCGEventFlagMaskCommand'), 62: ('ControlRight', 'kCGEventFlagMaskControl')}

In [ ]:
#| export
def trusted(): return available and bool(AXIsProcessTrusted())
def listening(): return available and bool(Q.CGPreflightListenEventAccess())
def status(): return dict(available=available, type=trusted(), listen=listening(), watching=_tap is not None)

In [ ]:
#| export
def prompt():
    "Show macOS's own permission prompts, and open the Settings pane for whichever is still missing."
    if not available: return
    if not trusted(): AXIsProcessTrustedWithOptions({'AXTrustedCheckOptionPrompt': True})
    if not listening(): Q.CGRequestListenEventAccess()
    for ok, pane in ((trusted(), 'Accessibility'), (listening(), 'ListenEvent')):
        if not ok: subprocess.run(['open', f'x-apple.systempreferences:com.apple.preference.security?Privacy_{pane}'])

## typing

In [ ]:
#| export
def _post(code, text=None):
    src = Q.CGEventSourceCreate(Q.kCGEventSourceStateHIDSystemState)
    for down in (True, False):
        e = Q.CGEventCreateKeyboardEvent(src, code, down)
        Q.CGEventSetFlags(e, 0)
        if text: Q.CGEventKeyboardSetUnicodeString(e, len(text.encode('utf-16-le')) // 2, text)
        Q.CGEventPost(Q.kCGHIDEventTap, e)

In [ ]:
#| export
def type_key(k):
    "Type a named special key, or the characters of `k`, into whichever app has focus. False if macOS won't allow it."
    if not trusted(): return False
    if k in CODES: _post(CODES[k])
    else: _post(0, k)
    return True

## watching the hold key

In [ ]:
#| export
_tap, _subs, _lock, _down = None, set(), threading.Lock(), set()

In [ ]:
#| export
def _emit(code, down):
    msg = f'{{"code":"{code}","down":{"true" if down else "false"}}}'
    for loop, q in list(_subs): loop.call_soon_threadsafe(q.put_nowait, msg)

In [ ]:
#| export
def _cb(proxy, typ, ev, _):
    if typ in (Q.kCGEventTapDisabledByTimeout, Q.kCGEventTapDisabledByUserInput): Q.CGEventTapEnable(_tap, True); return ev
    kc = Q.CGEventGetIntegerValueField(ev, Q.kCGKeyboardEventKeycode)
    if kc in MODS:
        name, flag = MODS[kc]
        down = bool(Q.CGEventGetFlags(ev) & getattr(Q, flag))  # a modifier's own event has its flag set only while it is down
        if down != (kc in _down):
            _down.add(kc) if down else _down.discard(kc)
            _emit(name, down)
    return ev

In [ ]:
#| export
def _run():
    global _tap
    t = Q.CGEventTapCreate(Q.kCGSessionEventTap, Q.kCGHeadInsertEventTap, Q.kCGEventTapOptionListenOnly, Q.CGEventMaskBit(Q.kCGEventFlagsChanged), _cb, None)
    if t is None: return  # no Input Monitoring
    _tap = t
    Q.CFRunLoopAddSource(Q.CFRunLoopGetCurrent(), Q.CFMachPortCreateRunLoopSource(None, t, 0), Q.kCFRunLoopCommonModes)
    Q.CGEventTapEnable(t, True)
    Q.CFRunLoopRun()

In [ ]:
#| export
def watch():
    "Start watching the modifier keys once (a no-op without Input Monitoring). True when it is running."
    with _lock:
        if _tap is None and listening(): threading.Thread(target=_run, daemon=True).start(); _wait()
    return _tap is not None

In [ ]:
#| export
def _wait():
    import time
    for _ in range(20):
        if _tap is not None: return
        time.sleep(.05)

In [ ]:
#| export
async def events():
    "Server-sent events: one line of JSON each time the hold key goes down or up, anywhere in macOS."
    sub = (asyncio.get_running_loop(), asyncio.Queue()); _subs.add(sub)
    try:
        yield 'retry: 2000\n\n'
        while True:
            try: yield f'data: {await asyncio.wait_for(sub[1].get(), 15)}\n\n'
            except asyncio.TimeoutError: yield ': keepalive\n\n'
    finally: _subs.discard(sub)